In [12]:
# =============================================================================
# CÉLULA 0: PARÂMETROS PARA PAPERMILL (GITHUB ACTIONS)
# =============================================================================
import os

# Parâmetros que podem ser sobrescritos via papermill
EMAIL_REMETENTE = os.environ.get('EMAIL_REMETENTE', 'ab11.94958191@gmail.com')
SENHA_APP = os.environ.get('GMAIL_APP_PASSWORD', '')

In [8]:
# =============================================================================
# CÉLULA 1: INSTALAÇÃO DE BIBLIOTECAS E IMPORTAÇÕES
# =============================================================================
!pip install yfinance pandas-ta optuna feedparser vaderSentiment --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime
import time
import warnings
warnings.filterwarnings("ignore")

# Análise de sentimento (VADER – leve e sem conflitos)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import feedparser

# Otimização (caso queira usar futuramente)
import optuna

print("✅ Bibliotecas instaladas e carregadas.")

✅ Bibliotecas instaladas e carregadas.


In [9]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (TÉCNICAS, SENTIMENTO, KELLY, ETC.)
# =============================================================================

# -----------------------------------------------------------------------------
# INDICADORES TÉCNICOS (GENÉRICOS)
# -----------------------------------------------------------------------------
def calcular_eficiencia_candle(df):
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low']
    range_total = range_total.replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta = df['Close'] > df['Open']
    baixa = df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia

def detectar_regime(df, janela=20):
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
    df_temp['adx'] = adx['ADX_14']
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty:
        return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33:
            return 0
        elif v > v_p67 or a > v_p67:
            return 2
        else:
            return 1
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series

def detectar_swing_low(df, janela=10):
    lows = df['Low'].values
    swing_lows = []
    for i in range(janela, len(lows) - janela):
        if lows[i] == min(lows[i-janela:i+janela+1]):
            swing_lows.append(lows[i])
    if swing_lows:
        return float(swing_lows[-1])
    else:
        return float(df['Low'].min())

def detectar_swing_high(df, janela=10):
    highs = df['High'].values
    swing_highs = []
    for i in range(janela, len(highs) - janela):
        if highs[i] == max(highs[i-janela:i+janela+1]):
            swing_highs.append(highs[i])
    if swing_highs:
        return float(swing_highs[-1])
    else:
        return float(df['High'].max())

def calcular_lta_pivos(df, janela_pivo=5):
    lows = df['Low'].values
    fundos = []
    for i in range(janela_pivo, len(lows) - janela_pivo):
        if lows[i] == min(lows[i-janela_pivo:i+janela_pivo+1]):
            fundos.append((i, lows[i]))
    if len(fundos) >= 2:
        (x1, y1), (x2, y2) = fundos[-2], fundos[-1]
        x_atual = len(lows) - 1
        inclinacao = (y2 - y1) / (x2 - x1)
        suporte_dinamico = y2 + inclinacao * (x_atual - x2)
        return suporte_dinamico
    else:
        return None

def calcular_ltb_pivos(df, janela_pivo=5):
    highs = df['High'].values
    topos = []
    for i in range(janela_pivo, len(highs) - janela_pivo):
        if highs[i] == max(highs[i-janela_pivo:i+janela_pivo+1]):
            topos.append((i, highs[i]))
    if len(topos) >= 2:
        (x1, y1), (x2, y2) = topos[-2], topos[-1]
        if y1 > y2:
            x_atual = len(highs) - 1
            inclinacao = (y2 - y1) / (x2 - x1)
            resistencia_dinamica = y2 + inclinacao * (x_atual - x2)
            return resistencia_dinamica
    return None

# -----------------------------------------------------------------------------
# PADRÕES DE CANDLE (DIÁRIO)
# -----------------------------------------------------------------------------
def detectar_padrao_altista(df_diario):
    if len(df_diario) < 3:
        return False
    ultimo = df_diario.iloc[-1]
    penultimo = df_diario.iloc[-2]
    corpo_ult = abs(ultimo['Close'] - ultimo['Open'])
    range_ult = ultimo['High'] - ultimo['Low']
    sombra_inf_ult = min(ultimo['Close'], ultimo['Open']) - ultimo['Low']
    sombra_sup_ult = ultimo['High'] - max(ultimo['Close'], ultimo['Open'])

    if range_ult > 0:
        if sombra_inf_ult >= 2 * corpo_ult and sombra_sup_ult <= 0.3 * corpo_ult:
            return True
    corpo_pen = abs(penultimo['Close'] - penultimo['Open'])
    if penultimo['Close'] < penultimo['Open'] and ultimo['Close'] > ultimo['Open']:
        if ultimo['Open'] <= penultimo['Close'] and ultimo['Close'] >= penultimo['Open']:
            return True
    if penultimo['Close'] < penultimo['Open'] and ultimo['Close'] > ultimo['Open']:
        meio_corpo_pen = (penultimo['Open'] + penultimo['Close']) / 2
        if ultimo['Open'] <= penultimo['Close'] and ultimo['Close'] >= meio_corpo_pen:
            return True
    return False

def detectar_padrao_baixista(df_diario):
    if len(df_diario) < 3:
        return False
    ultimo = df_diario.iloc[-1]
    penultimo = df_diario.iloc[-2]
    corpo_ult = abs(ultimo['Close'] - ultimo['Open'])
    range_ult = ultimo['High'] - ultimo['Low']
    sombra_sup_ult = ultimo['High'] - max(ultimo['Close'], ultimo['Open'])
    sombra_inf_ult = min(ultimo['Close'], ultimo['Open']) - ultimo['Low']

    if range_ult > 0:
        if sombra_sup_ult >= 2 * corpo_ult and sombra_inf_ult <= 0.3 * corpo_ult:
            return True
    corpo_pen = abs(penultimo['Close'] - penultimo['Open'])
    if penultimo['Close'] > penultimo['Open'] and ultimo['Close'] < ultimo['Open']:
        if ultimo['Open'] >= penultimo['Close'] and ultimo['Close'] <= penultimo['Open']:
            return True
    if penultimo['Close'] > penultimo['Open'] and ultimo['Close'] < ultimo['Open']:
        meio_corpo_pen = (penultimo['Open'] + penultimo['Close']) / 2
        if ultimo['Open'] >= penultimo['Close'] and ultimo['Close'] <= meio_corpo_pen:
            return True
    return False

# -----------------------------------------------------------------------------
# SENTIMENTO (VADER)
# -----------------------------------------------------------------------------
analyzer = SentimentIntensityAnalyzer()

def obter_sentimento_ticker(ticker):
    try:
        ticker_limpo = ticker.replace('.SA', '')
        url = f"https://news.google.com/rss/search?q={ticker_limpo}+B3&hl=pt-BR&gl=BR&ceid=BR:pt-419"
        feed = feedparser.parse(url)
        scores = []
        for entry in feed.entries[:3]:
            vs = analyzer.polarity_scores(entry.title)
            scores.append(vs['compound'])
        return np.mean(scores) if scores else 0.0
    except:
        return 0.0

# -----------------------------------------------------------------------------
# VOLUME ANORMAL (PIN SIMPLIFICADO)
# -----------------------------------------------------------------------------
def detectar_volume_anormal(df, periodo=20, limiar=1.5):
    if len(df) < periodo:
        return False
    vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
    vol_ultimo = df['Volume'].iloc[-1]
    return vol_ultimo >= vol_medio * limiar

# -----------------------------------------------------------------------------
# GESTÃO DE RISCO (FRACTIONAL KELLY)
# -----------------------------------------------------------------------------
def fractional_kelly(win_rate, payoff_ratio, frac=0.25):
    if payoff_ratio <= 0:
        return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    kelly = max(0.0, min(kelly, 0.25))
    return kelly * frac

print("✅ Funções auxiliares carregadas.")

✅ Funções auxiliares carregadas.


In [10]:
# =============================================================================
# CÉLULA 3: FUNÇÕES PRINCIPAIS – SWING TRADE E POSITION TRADE
# =============================================================================

# Parâmetros Otimizados
OTIMIZADO_SWING = {
    'atr_period': 14,
    'atr_mult': 1.8,
    'swing_window': 12,
    'lta_pivo_window': 6,
    'ltb_pivo_window': 6,
    'mm200_semanal': True,
    'mm200_diaria': True,
    'volume_limiar': 1.3
}

OTIMIZADO_POSITION = {
    'atr_period': 14,
    'atr_mult': 2.5,
    'swing_window': 24,          # 24 meses
    'lta_pivo_window': 12,
    'ltb_pivo_window': 12,
    'mm200_mensal': True,
    'volume_limiar': 1.2
}

# -----------------------------------------------------------------------------
# FUNÇÃO PARA SWING TRADE (MESES) – JÁ EXISTENTE
# -----------------------------------------------------------------------------
def analisar_swing_trade(ticker, df_w=None, df_d=None):
    try:
        if df_w is None:
            return None
        df_w = df_w.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_w.columns):
            rename_map = {'close':'Close', 'high':'High', 'low':'Low', 'open':'Open', 'volume':'Volume'}
            df_w.rename(columns={k:v for k,v in rename_map.items() if k in df_w.columns}, inplace=True)

        df_w.sort_index(inplace=True)
        if not isinstance(df_w.index, pd.DatetimeIndex):
            df_w.index = pd.to_datetime(df_w.index)

        df_w['Eficiencia'] = calcular_eficiencia_candle(df_w)
        df_w['Regime'] = detectar_regime(df_w)

        ultimo = df_w.iloc[-1]
        entrada = float(ultimo['Close'])
        if pd.isna(entrada) or entrada <= 0:
            return None

        recent_high = float(df_w['High'].rolling(window=min(52, len(df_w))).max().iloc[-1])
        recent_low  = float(df_w['Low'].rolling(window=min(52, len(df_w))).min().iloc[-1])

        reg_atual = int(ultimo['Regime']) if not pd.isna(ultimo['Regime']) else -1
        efic = round(float(ultimo['Eficiencia']), 2) if not pd.isna(ultimo['Eficiencia']) else None

        atr_series = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=OTIMIZADO_SWING['atr_period'])
        atr = float(atr_series.iloc[-1]) if not atr_series.empty and not pd.isna(atr_series.iloc[-1]) else 0.0

        # Padrão diário
        padrao_altista = True
        padrao_baixista = True
        if df_d is not None:
            try:
                df_d_local = df_d.copy()
                if not df_d_local.empty:
                    if isinstance(df_d_local.columns, pd.MultiIndex):
                        df_d_local.columns = df_d_local.columns.droplevel(1)
                    df_d_local.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
                    padrao_altista = detectar_padrao_altista(df_d_local)
                    padrao_baixista = detectar_padrao_baixista(df_d_local)
            except:
                pass

        sentimento = obter_sentimento_ticker(ticker)
        volume_anormal = detectar_volume_anormal(df_w, periodo=20, limiar=OTIMIZADO_SWING['volume_limiar'])

        setups = []

        # COMPRA
        if padrao_altista:
            stop_atr = entrada - (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            try:
                sw_low = detectar_swing_low(df_w, janela=OTIMIZADO_SWING['swing_window'])
                stop_swing = sw_low if sw_low < entrada else None
            except:
                stop_swing = None
            try:
                lta = calcular_lta_pivos(df_w, janela_pivo=OTIMIZADO_SWING['lta_pivo_window'])
                stop_lta = lta if (lta is not None and lta < entrada) else None
            except:
                stop_lta = None

            for metodo, stop_loss in [('ATR', stop_atr), ('Swing Low', stop_swing), ('LTA Pivôs', stop_lta)]:
                if stop_loss is None or stop_loss <= 0 or stop_loss >= entrada:
                    continue
                risco = entrada - stop_loss
                alvo = entrada + (risco * 3)
                if alvo <= recent_high * 1.05:
                    setups.append({
                        'Ticker': ticker,
                        'Modalidade': 'Swing',
                        'Direcao': 'COMPRA',
                        'Entrada': round(entrada, 2),
                        'Método Stop': metodo,
                        'Stop Loss': round(stop_loss, 2),
                        'Risco (R$)': round(risco, 2),
                        'Alvo 3:1': round(alvo, 2),
                        'Resistência': round(recent_high, 2),
                        'Suporte': round(recent_low, 2),
                        'Regime': reg_atual,
                        'Eficiência': efic,
                        'Sentimento': round(sentimento, 2),
                        'Volume Anormal': volume_anormal
                    })

        # VENDA
        if padrao_baixista:
            stop_atr = entrada + (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            try:
                sw_high = detectar_swing_high(df_w, janela=OTIMIZADO_SWING['swing_window'])
                stop_swing = sw_high if sw_high > entrada else None
            except:
                stop_swing = None
            try:
                ltb = calcular_ltb_pivos(df_w, janela_pivo=OTIMIZADO_SWING['ltb_pivo_window'])
                stop_ltb = ltb if (ltb is not None and ltb > entrada) else None
            except:
                stop_ltb = None

            for metodo, stop_loss in [('ATR', stop_atr), ('Swing High', stop_swing), ('LTB Pivôs', stop_ltb)]:
                if stop_loss is None or stop_loss <= entrada:
                    continue
                risco = stop_loss - entrada
                alvo = entrada - (risco * 3)
                if alvo >= recent_low * 0.95:
                    setups.append({
                        'Ticker': ticker,
                        'Modalidade': 'Swing',
                        'Direcao': 'VENDA',
                        'Entrada': round(entrada, 2),
                        'Método Stop': metodo,
                        'Stop Loss': round(stop_loss, 2),
                        'Risco (R$)': round(risco, 2),
                        'Alvo 3:1': round(alvo, 2),
                        'Resistência': round(recent_high, 2),
                        'Suporte': round(recent_low, 2),
                        'Regime': reg_atual,
                        'Eficiência': efic,
                        'Sentimento': round(sentimento, 2),
                        'Volume Anormal': volume_anormal
                    })

        return setups if setups else None
    except:
        return None

# -----------------------------------------------------------------------------
# FUNÇÃO PARA POSITION TRADE (ANOS)
# -----------------------------------------------------------------------------
def analisar_position_trade(ticker, df_m=None):
    try:
        if df_m is None:
            return None
        df_m = df_m.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_m.columns):
            rename_map = {'close':'Close', 'high':'High', 'low':'Low', 'open':'Open', 'volume':'Volume'}
            df_m.rename(columns={k:v for k,v in rename_map.items() if k in df_m.columns}, inplace=True)

        df_m.sort_index(inplace=True)
        if not isinstance(df_m.index, pd.DatetimeIndex):
            df_m.index = pd.to_datetime(df_m.index)

        df_m['Eficiencia'] = calcular_eficiencia_candle(df_m)
        df_m['Regime'] = detectar_regime(df_m)

        ultimo = df_m.iloc[-1]
        entrada = float(ultimo['Close'])
        if pd.isna(entrada) or entrada <= 0:
            return None

        # Resistência e Suporte de 5 anos (60 meses)
        lookback = min(60, len(df_m))
        recent_high = float(df_m['High'].rolling(window=lookback).max().iloc[-1])
        recent_low  = float(df_m['Low'].rolling(window=lookback).min().iloc[-1])

        reg_atual = int(ultimo['Regime']) if not pd.isna(ultimo['Regime']) else -1
        efic = round(float(ultimo['Eficiencia']), 2) if not pd.isna(ultimo['Eficiencia']) else None

        atr_series = ta.atr(df_m['High'], df_m['Low'], df_m['Close'], length=OTIMIZADO_POSITION['atr_period'])
        atr = float(atr_series.iloc[-1]) if not atr_series.empty and not pd.isna(atr_series.iloc[-1]) else 0.0

        sentimento = obter_sentimento_ticker(ticker)
        volume_anormal = detectar_volume_anormal(df_m, periodo=20, limiar=OTIMIZADO_POSITION['volume_limiar'])

        setups = []

        # COMPRA
        stop_atr = entrada - (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        try:
            sw_low = detectar_swing_low(df_m, janela=OTIMIZADO_POSITION['swing_window'])
            stop_swing = sw_low if sw_low < entrada else None
        except:
            stop_swing = None
        try:
            lta = calcular_lta_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['lta_pivo_window'])
            stop_lta = lta if (lta is not None and lta < entrada) else None
        except:
            stop_lta = None

        for metodo, stop_loss in [('ATR', stop_atr), ('Swing Low', stop_swing), ('LTA Pivôs', stop_lta)]:
            if stop_loss is None or stop_loss <= 0 or stop_loss >= entrada:
                continue
            risco = entrada - stop_loss
            alvo = entrada + (risco * 3)
            if alvo <= recent_high * 1.10:   # tolerância maior para position
                setups.append({
                    'Ticker': ticker,
                    'Modalidade': 'Position',
                    'Direcao': 'COMPRA',
                    'Entrada': round(entrada, 2),
                    'Método Stop': metodo,
                    'Stop Loss': round(stop_loss, 2),
                    'Risco (R$)': round(risco, 2),
                    'Alvo 3:1': round(alvo, 2),
                    'Resistência': round(recent_high, 2),
                    'Suporte': round(recent_low, 2),
                    'Regime': reg_atual,
                    'Eficiência': efic,
                    'Sentimento': round(sentimento, 2),
                    'Volume Anormal': volume_anormal
                })

        # VENDA
        stop_atr = entrada + (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        try:
            sw_high = detectar_swing_high(df_m, janela=OTIMIZADO_POSITION['swing_window'])
            stop_swing = sw_high if sw_high > entrada else None
        except:
            stop_swing = None
        try:
            ltb = calcular_ltb_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['ltb_pivo_window'])
            stop_ltb = ltb if (ltb is not None and ltb > entrada) else None
        except:
            stop_ltb = None

        for metodo, stop_loss in [('ATR', stop_atr), ('Swing High', stop_swing), ('LTB Pivôs', stop_ltb)]:
            if stop_loss is None or stop_loss <= entrada:
                continue
            risco = stop_loss - entrada
            alvo = entrada - (risco * 3)
            if alvo >= recent_low * 0.90:
                setups.append({
                    'Ticker': ticker,
                    'Modalidade': 'Position',
                    'Direcao': 'VENDA',
                    'Entrada': round(entrada, 2),
                    'Método Stop': metodo,
                    'Stop Loss': round(stop_loss, 2),
                    'Risco (R$)': round(risco, 2),
                    'Alvo 3:1': round(alvo, 2),
                    'Resistência': round(recent_high, 2),
                    'Suporte': round(recent_low, 2),
                    'Regime': reg_atual,
                    'Eficiência': efic,
                    'Sentimento': round(sentimento, 2),
                    'Volume Anormal': volume_anormal
                })

        return setups if setups else None
    except:
        return None

print("✅ Célula 3 carregada (Swing + Position).")

✅ Célula 3 carregada (Swing + Position).


In [11]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL – SWING E POSITION TRADE
#            (CORRIGIDA PARA GITHUB ACTIONS E COLAB)
# =============================================================================

# --- E-mail (usa variáveis definidas na Célula 0) ---
# Tenta obter do userdata do Colab se estiver vazio (fallback para execução manual)
try:
    from google.colab import userdata
    if not SENHA_APP:
        SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except:
    pass  # Mantém o que já foi definido (via environment ou vazio)

# --- Filtros de liquidez e qualidade (comuns) ---
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
PRECO_MINIMO = 5.00
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.20
EXIGIR_CONFLUENCIA = True

# Kelly
CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
FRACAO_KELLY = 0.25

# Setores bloqueados
SETORES_BLOQUEADOS = ['VEST', 'MODA', 'CALCADO', 'AEREA', 'EDUCAC', 'SAUDE', 'HOSP']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA',
                      'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']

# --- Função de e-mail (genérica) ---
def enviar_email_ou_exibir(oportunidades, modalidade):
    if not oportunidades:
        print(f"ℹ️ Nenhuma oportunidade de {modalidade} encontrada.")
        return
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            msg['Subject'] = f"🚨 Oportunidades {modalidade} 3:1 - {datetime.now().strftime('%d/%m/%Y')}"
            corpo = f"<h2>Setups {modalidade} Detectados</h2><ul>"
            for op in oportunidades:
                corpo += f"<li><b>{op['Ticker']} ({op['Direcao']})</b> | Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f} | Alvo: R$ {op['Alvo 3:1']:.2f} | Lote: {op['Lote']} ações</li>"
            corpo += "</ul>"
            msg.attach(MIMEText(corpo, 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            print(f"✅ E-mail ({modalidade}) enviado para {EMAIL_REMETENTE}")
        except Exception as e:
            print(f"❌ Falha no e-mail ({modalidade}): {e}")
    else:
        print(f"📧 E-mail não configurado. Exibindo {modalidade} na tela.\n")

    # Exibição da tabela
    df_op = pd.DataFrame(oportunidades)
    colunas = ['Ticker', 'Direcao', 'Entrada', 'Método Stop', 'Stop Loss', 'Risco (R$)',
               'Alvo 3:1', 'Resistência', 'Suporte', 'Regime', 'Eficiência',
               'Sentimento', 'Volume Anormal', 'Kelly %', 'Lote']
    display(df_op[colunas].sort_values(['Direcao', 'Ticker']))

    # Download do CSV
    csv_name = f"oportunidades_{modalidade.lower()}_{datetime.now().strftime('%Y%m%d')}.csv"
    df_op[colunas].to_csv(csv_name, index=False)
    from google.colab import files
    files.download(csv_name)
    print(f"\n📁 Arquivo '{csv_name}' baixado.")

# --- Obter tickers (com fallback) ---
def obter_tickers_b3():
    try:
        url = "https://www.dadosdemercado.com.br/acoes"
        soup = BeautifulSoup(requests.get(url, timeout=10).content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cols = row.find_all('td')
            if cols:
                t = cols[0].text.strip()
                if t and not t.startswith('#'):
                    tickers.append(t)
        if tickers:
            return tickers
    except:
        pass
    # Fallback estático
    return ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4']

print("🔍 Obtendo tickers...")
tickers_b3 = obter_tickers_b3()
print(f"✅ {len(tickers_b3)} tickers.")

# --- Filtro de liquidez ---
print("💧 Filtrando liquidez...")
tickers_liquidos = []
tickers_yahoo = [t + ".SA" for t in tickers_b3]

for i in range(0, len(tickers_yahoo), 50):
    chunk = tickers_yahoo[i:i+50]
    try:
        infos = yf.Tickers(' '.join(chunk)).tickers
        for ty in chunk:
            try:
                info = infos[ty].info
                vol = info.get('averageVolume', 0)
                pr = info.get('regularMarketPreviousClose', 0)
                if vol >= VOLUME_MINIMO_ACAO and vol * pr >= VOLUME_FINANCEIRO_MINIMO:
                    nome = info.get('longName', '').upper()
                    if not any(p in nome for p in SETORES_BLOQUEADOS):
                        tickers_liquidos.append(ty)
            except:
                continue
    except:
        continue
    time.sleep(0.5)

print(f"💧 {len(tickers_liquidos)} ativos líquidos.")

# --- DOWNLOADS EM LOTE (SEMANAL, DIÁRIO, MENSAL) ---
print("📦 Baixando dados semanais em lote...")
data_w = yf.download(tickers_liquidos, period='2y', interval='1wk', group_by='ticker', progress=False, auto_adjust=True)

print("📦 Baixando dados diários em lote...")
data_d = yf.download(tickers_liquidos, period='1y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)

print("📦 Baixando dados mensais em lote (máximo histórico)...")
data_m = yf.download(tickers_liquidos, period='max', interval='1mo', group_by='ticker', progress=False, auto_adjust=True)

# --- Parâmetros para Position Trade (MM50 mensal) ---
OTIMIZADO_POSITION = {
    'atr_period': 14,
    'atr_mult': 2.5,
    'swing_window': 24,
    'lta_pivo_window': 12,
    'ltb_pivo_window': 12,
    'mm50_mensal': True,
    'volume_limiar': 1.2
}

# --- Processamento ---
kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, FRACAO_KELLY)
risco_maximo = CAPITAL_TOTAL * kelly_pct

oportunidades_swing = []
oportunidades_position = []

for i, ticker in enumerate(tickers_liquidos):
    print(f"Analisando {ticker} ({i+1}/{len(tickers_liquidos)})...", end='\r')
    if ticker in TICKERS_BLOQUEADOS:
        continue

    def get_df(data, ticker):
        if data is not None and ticker in data:
            df = data[ticker].copy()
            df.columns = [col.lower() for col in df.columns]
            df.rename(columns={'close':'Close', 'high':'High', 'low':'Low', 'open':'Open', 'volume':'Volume'}, inplace=True)
            return df
        return None

    df_w = get_df(data_w, ticker)
    df_d = get_df(data_d, ticker)
    df_m = get_df(data_m, ticker)

    # === SWING TRADE ===
    if df_w is not None and not df_w.empty:
        res_swing = analisar_swing_trade(ticker, df_w=df_w, df_d=df_d)
        if res_swing:
            for r in res_swing:
                e = r['Entrada']
                risco_pct = r['Risco (R$)'] / e
                if e < PRECO_MINIMO: continue
                if risco_pct < RISCO_PERCENTUAL_MINIMO: continue
                if risco_pct > RISCO_PERCENTUAL_MAXIMO: continue
                if EXIGIR_CONFLUENCIA:
                    if r['Regime'] not in [1,2]: continue
                    if r['Eficiência'] is None or r['Eficiência'] < 0.6: continue
                # MM200 Semanal
                if OTIMIZADO_SWING['mm200_semanal']:
                    mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
                    if pd.isna(mm200w): continue
                    if (r['Direcao'] == 'COMPRA' and e < mm200w) or (r['Direcao'] == 'VENDA' and e > mm200w):
                        continue
                # MM200 Diária
                if OTIMIZADO_SWING['mm200_diaria'] and df_d is not None:
                    mm200d = df_d['Close'].rolling(200).mean().iloc[-1]
                    if pd.isna(mm200d): continue
                    if (r['Direcao'] == 'COMPRA' and e < mm200d) or (r['Direcao'] == 'VENDA' and e > mm200d):
                        continue

                lote = int(risco_maximo / r['Risco (R$)'])
                lote = (lote // 100) * 100
                if lote == 0: continue
                r['Kelly %'] = round(kelly_pct * 100, 2)
                r['Lote'] = lote
                oportunidades_swing.append(r)

    # === POSITION TRADE ===
    if df_m is not None and not df_m.empty:
        res_pos = analisar_position_trade(ticker, df_m=df_m)
        if res_pos:
            for r in res_pos:
                e = r['Entrada']
                risco_pct = r['Risco (R$)'] / e
                if e < PRECO_MINIMO: continue
                if risco_pct < 0.03: continue
                if risco_pct > 0.30: continue
                if EXIGIR_CONFLUENCIA:
                    if r['Regime'] not in [1,2]: continue
                    if r['Eficiência'] is None or r['Eficiência'] < 0.5: continue
                # MM50 Mensal
                if OTIMIZADO_POSITION['mm50_mensal']:
                    mm50m = df_m['Close'].rolling(50).mean().iloc[-1]
                    if pd.isna(mm50m): continue
                    if (r['Direcao'] == 'COMPRA' and e < mm50m) or (r['Direcao'] == 'VENDA' and e > mm50m):
                        continue

                lote = int(risco_maximo / r['Risco (R$)'])
                lote = (lote // 100) * 100
                if lote == 0: continue
                r['Kelly %'] = round(kelly_pct * 100, 2)
                r['Lote'] = lote
                oportunidades_position.append(r)

print(f"\n🎯 Swing Trade: {len(oportunidades_swing)} setups | Position Trade: {len(oportunidades_position)} setups")
print(f"   Kelly recomendado: {kelly_pct*100:.2f}% do capital (R$ {risco_maximo:.2f})")

# --- Exibir resultados ---
if oportunidades_swing:
    enviar_email_ou_exibir(oportunidades_swing, "Swing Trade")
else:
    print("\n⚪ Nenhum setup de Swing Trade hoje.")

if oportunidades_position:
    enviar_email_ou_exibir(oportunidades_position, "Position Trade")
else:
    print("\n⚪ Nenhum setup de Position Trade hoje.")

🔍 Obtendo tickers...
✅ 391 tickers.
💧 Filtrando liquidez...
💧 140 ativos líquidos.
📦 Baixando dados semanais em lote...
📦 Baixando dados diários em lote...
📦 Baixando dados mensais em lote (máximo histórico)...

🎯 Swing Trade: 0 setups | Position Trade: 0 setups
   Kelly recomendado: 5.00% do capital (R$ 5000.00)

⚪ Nenhum setup de Swing Trade hoje.

⚪ Nenhum setup de Position Trade hoje.
